In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

#Importando os arquivos de pre-processamento e download de dados
from pre_processamento import *
from data_downloading import *
from collections import Counter
from vector_model import *
from evaluation import *
from recuperar import *


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/agnesmva/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#Download dos dados
colecao_subset_removida = download_data_subset_removida()
colecao_subset_query = download_data_subset_query()

Carregando colecao_subset_removida.pkl do cache local...
Carregando colecao_subset_query.pkl do cache local...


In [ ]:
#O processo de lematização é aplicado a cada documento da coleção, resultando em uma nova lista de documentos lematizados.
#Atualmente é o processo mais demorado de pre-processamento. 
#Resolvi dividir o processo de pre-processamento em etapas para facilitar a identificação de gargalos e otimizar o código posteriormente, se necessário.
lema_subset = [preprocess_lemma(doc) for doc in colecao_subset_removida.data]

In [4]:
steaming_subset = [preprocess_stemming(doc) for doc in colecao_subset_removida.data]

In [5]:
original_subset = [preprocess_original(doc) for doc in colecao_subset_removida.data]

In [6]:
print("Original:")
print(sum(len(doc) for doc in original_subset))

print("Steamming:")
print(sum(len(doc) for doc in steaming_subset))

print("Lema:")
print(sum(len(doc) for doc in lema_subset))

Original:
3423145
Steamming:
1765680
Lema:
1473364


Vamos verificar os termos únicos após o processo de pré-processamento

In [7]:
vocab_original = set(token for doc in original_subset for token in doc)
vocab_stemming = set(token for doc in steaming_subset for token in doc)
vocab_lemma = set(token for doc in lema_subset for token in doc)

print(f"Original: {len(vocab_original):,}")
print(f"Stemming: {len(vocab_stemming):,}")
print(f"Lematização: {len(vocab_lemma):,}")

Original: 260,121
Stemming: 72,121
Lematização: 63,746


In [8]:
orig = len(vocab_original)
stem = len(vocab_stemming)
lemma = len(vocab_lemma)

print(f"Redução Stemming: {(orig-stem)/orig*100:.2f}%")
print(f"Redução Lematização: {(orig-lemma)/orig*100:.2f}%")

Redução Stemming: 72.27%
Redução Lematização: 75.49%


Enquanto executávamos o pre-processamento resolvemos verificar como ficaram as 20 primeiras palavras que mais apareciam. Foi bom, pois conseguimos identificar problemas na nossa função de stemming. O curioso é que ficou como termo mais retornando a letra X, não sabemos o porquê.

In [9]:
freq_original = Counter(
    token
    for doc in original_subset
    for token in doc
)

freq_stemming = Counter(
    token
    for doc in steaming_subset
    for token in doc
)

freq_lemma = Counter(
    token
    for doc in lema_subset
    for token in doc
)

print(freq_original.most_common(20))
print(freq_stemming.most_common(20))
print(freq_lemma.most_common(20))

[('the', 171107), ('to', 85388), ('of', 76423), ('a', 68859), ('and', 67873), ('in', 49183), ('is', 47086), ('i', 45586), ('that', 42333), ('for', 31420), ('it', 28422), ('you', 26125), ('on', 22342), ('be', 21462), ('this', 21109), ('have', 20914), ('with', 20277), ('are', 20178), ('not', 18820), ('as', 18179)]
[('use', 11518), ('one', 10990), ('would', 10192), ('get', 7455), ('like', 7369), ('peopl', 6523), ('know', 6466), ('time', 6074), ('also', 5592), ('think', 5580), ('make', 5010), ('say', 4988), ('max', 4750), ('work', 4719), ('year', 4364), ('system', 4344), ('file', 4211), ('could', 4181), ('well', 4128), ('new', 4109)]
[('know', 7537), ('like', 6656), ('people', 6515), ('think', 6328), ('time', 5748), ('use', 5582), ('good', 5228), ('say', 4753), ('work', 4539), ('year', 4321), ('new', 4222), ('go', 4152), ('file', 4094), ('come', 3995), ('want', 3976), ('system', 3975), ('look', 3833), ('find', 3797), ('way', 3762), ('need', 3748)]


Agora, após o pré-processamento do corpus. 

In [10]:
vocabulary, term_to_idx = build_vocabulary(lema_subset)
print(f"Vocabulário: {len(vocabulary)} termos")
print(vocabulary[:5])
print(term_to_idx)

Vocabulário: 63746 termos
['aaa', 'aaaaa', 'aaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaauuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuugggggggggggggggg', 'aaaaarrrrgh']
{'aaa': 0, 'aaaaa': 1, 'aaaaaaaaaaaa': 2, 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaauuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuugggggggggggggggg': 3, 'aaaaarrrrgh': 4, 'aaaall': 5, 'aaack': 6, 'aaaggghhh': 7, 'aaah': 8, 'aaahh': 9, 'aaahhhh': 10, 'aaai': 11, 'aacc': 12, 'aacs': 13, 'aagain': 14, 'aah': 15, 'aalternate': 16, 'aam': 17, 'aamazing': 18, 'aammmaaaazzzzzziinnnnggggg': 19, 'aan': 20, 'aanbieden': 21, 'aand': 22, 'aanerud': 23, 'aangeboden': 24, 'aangegeven': 25, 'aangezien': 26, 'aantal': 27, 'aao': 28, 'aap': 29, 'aaplay': 30, 'aargh': 31, 'aarhus': 32, 'aarnet': 33, 'aaron': 34, 'aaronson': 35, 'aaroundpluto': 36, 'aarp': 37, 'aarseth': 38, 'aarskog': 39, 'aas': 40, 'aaske': 41, 'aat': 42, 'aatdb': 43, 'aavso': 44, 'aawin': 45, 'ab': 46, 'ababa': 47, 'ababs': 48, 'abandon': 49, 'abandond': 50, 'abandondone': 51, 'abandonment': 52, 'abate': 

In [11]:
# --- Passo 1: Calcular DF ---
df_vector = calcular_frequencia_documento(lema_subset, vocabulary, term_to_idx)
print(f"\nDF para os 5 primeiros termos: {df_vector[:5]}")

# --- Passo 2: Calcular IDF ---
idf_vector = calcular_idf(df_vector, total_documentos=len(lema_subset))
print(f"\nIDF para os 5 primeiros termos: {idf_vector[:5]}")

# --- Passo 3: Construir as Matrizes Finais ---
tf_matrix, tfidf_matrix = calcular_tf_idf_matriz(lema_subset, vocabulary, term_to_idx, idf_vector)
print(f"\nFormato da matriz TF-IDF: {len(tfidf_matrix)} documentos x {len(vocabulary)} colunas.")


DF para os 5 primeiros termos: [29, 2, 1, 1, 1]

IDF para os 5 primeiros termos: [7.442911647354473, 9.745496740348518, 10.150961848456683, 10.150961848456683, 10.150961848456683]

Formato da matriz TF-IDF: 18846 documentos x 63746 colunas.


Pra testar a função de calcular corretamente o TF-IDF da query coloquei esse exemplo. 
ela retorna em todo o documento, e no inicio do documento estão indexados termos como aaa aaa, etc. que não servem pra nossa busca, por isso coloquei o if para retornar o tf-idf no termo médio e retorna esses valores.

In [12]:
query = "There's so many things that you could be, but are you in it for the love or are you just in it for the money?"
query_tokens = preprocess_lemma(query)
query_vector = calcular_tf_idf_query(query_tokens, vocabulary, term_to_idx, idf_vector)
for query_term, value in zip(vocabulary, query_vector):
    if value > 0:
        print(f"Termo: '{query_term}', TF-IDF: {value:.4f}")

Termo: 'love', TF-IDF: 4.6355
Termo: 'money', TF-IDF: 4.4439
Termo: 'thing', TF-IDF: 3.0468


In [ ]:
query_vector = normalize(query_vector)

ranking = rank_documents(
    query_vector,
    tfidf_matrix,
)

print(ranking[:10])

[(6458, 0.39252857979220057), (7978, 0.3812357773788567), (4884, 0.3667032175225678), (10406, 0.2997516218408424), (5488, 0.26498097626970235), (10965, 0.25990221543765246), (592, 0.24502703400912842), (2029, 0.23189410417757292), (8179, 0.22886841626532545), (9447, 0.22501168885710301)]


In [ ]:
resultados_alinhados = buscar_documentos(query_vector, tfidf_matrix, 3)
print(f"Resultados para a busca: '{query}'\n" + "="*50)

for posicao, (idx, score) in enumerate(resultados_alinhados, 1):
    print(f"\n{posicao}º Lugar - Documento Índice [{idx}] | Score de Similaridade: {score:.4f}")
    print("-"*50)
    
    texto_original = colecao_subset_removida.data[idx]
    
    print(texto_original[:300] + "...")

Resultados para a busca: 'There's so many things that you could be, but are you in it for the love or are you just in it for the money?'

1º Lugar - Documento Índice [6458] | Score de Similaridade: 0.3925
--------------------------------------------------
So long as we think that good things are what we *have* to do rather than
what we come to *want* to do, we miss the point. The more we love God; the
more we come to love what and whom He loves.

When I find that what I am doing is not good, it is not a sign to try
even harder (Romans 7:14-8:2); it i...

2º Lugar - Documento Índice [7978] | Score de Similaridade: 0.3812
--------------------------------------------------
Above all, love each other deeply, because love covers over a multitude of
sins. ...

3º Lugar - Documento Índice [4884] | Score de Similaridade: 0.3667
--------------------------------------------------
davem@bnr.ca (Dave Mielke) writes,

 
  
I am extremely uncomfortable with this way of phrasing it.  God's love 
is u

In [ ]:
textos_base = colecao_subset_removida.data
N_base = len(textos_base)

In [ ]:
# ==========================================
# MOTOR 2: STEMMING
# ==========================================
print("Construindo Base com Stemming...")
# Aplica a função em todos os documentos da base
docs_stem = steaming_subset 

# Gera os componentes do TF-IDF
vocab_stem, idx_stem = build_vocabulary(docs_stem)
df_stem = calcular_frequencia_documento(docs_stem, vocab_stem, idx_stem)
idf_stem = calcular_idf(df_stem, N_base)
_, matrix_stem = calcular_tf_idf_matriz(docs_stem, vocab_stem, idx_stem, idf_stem)


In [ ]:
# ==========================================
# MOTOR 3: LEMATIZAÇÃO (SpaCy)
# ==========================================
print("Construindo Base com Lematização...")
# Aplica a função em todos os documentos da base
docs_lemma = lema_subset

# Gera os componentes do TF-IDF
vocab_lemma, idx_lemma = build_vocabulary(docs_lemma)
df_lemma = calcular_frequencia_documento(docs_lemma, vocab_lemma, idx_lemma)
idf_lemma = calcular_idf(df_lemma, N_base)
_, matrix_lemma = calcular_tf_idf_matriz(docs_lemma, vocab_lemma, idx_lemma, idf_lemma)


print("\nPronto! Todos os vocabulários e matrizes TF-IDF foram gerados com sucesso.")

In [ ]:
# 1. Separamos os textos da base de dados que servirá como nosso "Google"
textos_base = colecao_subset_removida.data
N_base = len(textos_base)

print(f"Indexando {N_base} documentos. Isso pode levar um tempinho...")

# ==========================================
# MOTOR 1: TEXTO ORIGINAL (Apenas Tokenização)
# ==========================================
#print("\nConstruindo Base Original...")
# Aplica a função em todos os documentos da base
#docs_orig = original_subset

# Gera os componentes do TF-IDF
#vocab_orig, idx_orig = build_vocabulary(docs_orig)
#df_orig = calcular_frequencia_documento(docs_orig, vocab_orig, idx_orig)
#idf_orig = calcular_idf(df_orig, N_base)
#_, matrix_orig = calcular_tf_idf_matriz(docs_orig, vocab_orig, idx_orig, idf_orig)






Indexando 18846 documentos. Isso pode levar um tempinho...

Construindo Base Original...


Agora vamos calcular algumas métricas para compreender como ficaram os dados.
1. Calcular AP
2. Calcular MAP
3. Calcular Precision@K
4. Calcular Recall@K

In [ ]:
import json

# --- 1. Configuração Inicial ---
categorias_para_avaliar = ["comp.graphics", "sci.med", "rec.sport.baseball"]
LIMITAR_QUERIES = 50

# Suas queries (usando o seu subset de testes)
queries_textos = colecao_subset_query.data[:LIMITAR_QUERIES]
queries_targets = colecao_subset_query.target[:LIMITAR_QUERIES]

# Sua base de conhecimento (usando o seu subset de treino)
base_targets = colecao_subset_removida.target

# O dicionário que vai guardar tudo
results = {}

# --- 2. Avaliação dos Pipelines ---

print("Avaliando pipeline Original...")
results["original"] = {
    "AP": avaliar_recuperacao_por_categoria(
        queries_textos=queries_textos, 
        queries_targets=queries_targets, 
        base_targets=base_targets,
        funcao_preprocessamento=preprocess_original, # <- Coloque sua função de pré-processamento original aqui
        vocabulary=vocab_original,                       # <- Coloque seu vocabulário original aqui
        term_to_idx=idx_orig,                        # <- Coloque seu term_to_idx original aqui
        idf_vector=idf_orig,                         # <- Coloque seu vetor IDF original aqui
        tfidf_matrix=matrix_orig,                    # <- Coloque sua matriz TF-IDF original aqui
        categorias_alvo=categorias_para_avaliar, 
        nomes_categorias=colecao_subset_query.target_names
    )
}

print("Avaliando pipeline Stemming...")
results["stemming"] = {
    "AP": avaliar_recuperacao_por_categoria(
        queries_textos=queries_textos, 
        queries_targets=queries_targets, 
        base_targets=base_targets,
        funcao_preprocessamento=preprocess_stemming,     # <- Coloque sua função com stemming aqui
        vocabulary=vocab_stemming, 
        term_to_idx=idx_stem, 
        idf_vector=idf_stem, 
        tfidf_matrix=matrix_stem,
        categorias_alvo=categorias_para_avaliar, 
        nomes_categorias=colecao_subset_query.target_names
    )
}

print("Avaliando pipeline Lemma...")
results["lemma"] = {
    "AP": avaliar_recuperacao_por_categoria(
        queries_textos=queries_textos, 
        queries_targets=queries_targets, 
        base_targets=base_targets,
        funcao_preprocessamento=preprocess_lemma,    # <- Coloque sua função com lematização aqui
        vocabulary=vocab_lemma, 
        term_to_idx=idx_lemma, 
        idf_vector=idf_lemma, 
        tfidf_matrix=matrix_lemma,
        categorias_alvo=categorias_para_avaliar, 
        nomes_categorias=colecao_subset_query.target_names
    )
}

# --- 3. Exibição dos Resultados ---
print("\nResultados Finais:")
print(json.dumps(results, indent=4))

Avaliando pipeline Original...

Resultados Finais:
{}
